# LiteDiseaseNetV4 - custom architecture, 38 classes

A custom network rather than a stock backbone, trained on the same repaired split as
`crop_disease_full_pipeline.ipynb`.

**Run Part A of that notebook first.** This one reads the `manifest_clean.csv` it
writes; it does not repair or split anything itself. Results land in the same folder,
so the comparison table in that notebook picks them up automatically.

### The architecture

MobileNetV2 is split in two and the image is read at two scales at once:

| Branch | Sees | Why |
|---|---|---|
| **Texture** | `features[:7]`, 32ch @ 28x28 | lesion texture and edges, before detail is pooled away |
| **Semantic** | `features[7:]`, 1280ch @ 7x7 | what the leaf and the disease *are* |

Both branches carry **CBAM** attention - channel attention ("which features matter")
*and* spatial attention ("where in the leaf matters"). Lesion position and shape is
often exactly what separates two confusable diseases, and a channel-only SE block
cannot express it. The branches are combined by a **learned gate** that reweights them
per image rather than by plain concatenation.

### The recipe, and why it differs

- **Discriminative learning rates.** The pretrained backbone moves at 3e-5, the new
  modules at 3e-4: randomly-initialised layers need to travel much further than weights
  that already encode useful features.
- **Label smoothing 0.1.** Stops the model driving every logit to a hard 0/1. This is
  what keeps its confidences honest, and it is why the confidence gate at the end of
  this notebook actually separates out-of-scope photos - a cross-entropy model rates
  leaves it has never seen at 100%.
- **AdamW with weight decay 1e-4**, and the checkpoint chosen on **val macro-F1**, not
  accuracy, because macro-F1 gives every class one vote under heavy imbalance.

In [ ]:
# ---- config ---------------------------------------------------------------- #
SEED, EPOCHS, BATCH_SIZE, IMG_SIZE = 42, 25, 32, 224
LR_BACKBONE, LR_HEAD = 3e-5, 3e-4
LABEL_SMOOTHING, WEIGHT_DECAY = 0.1, 1e-4
USE_CLASS_WEIGHTS, USE_AMP = True, True

import json, os, random, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
from tqdm.auto import tqdm

OUT_DIR = Path.cwd() / "crop-disease-38"
MANIFEST = OUT_DIR / "manifest_clean.csv"
assert MANIFEST.is_file(), (
    f"{MANIFEST} missing - run Part A of crop_disease_full_pipeline.ipynb first")

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ON = USE_AMP and DEVICE.type == "cuda"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# where the images actually live, for the stale-path fallback below
DATA_ROOT = Path("plantvillage_full/raw/color")

clean = pd.read_csv(MANIFEST, low_memory=False)
clean = clean[clean["status"] == "ok"] if "status" in clean else clean
CLASS_NAMES = sorted(clean["class_raw"].unique())
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

# stored absolute paths may be stale if data was re-extracted - remap by filename
if not all(Path(p).is_file() for p in clean["path"].head(50)):
    index = {}
    for dirpath, dirnames, filenames in os.walk(DATA_ROOT):
        for f in filenames:
            index.setdefault(f, str(Path(dirpath) / f))
    clean = clean.assign(path=[index[f] for f in clean["filename"]])
    print("remapped stale image paths")

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.0, hue=0.0),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(round(IMG_SIZE * 8 / 7)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class ManifestDataset(Dataset):
    def __init__(self, frame, transform):
        self.samples = [(r.path, CLASS_TO_IDX[r.class_raw])
                        for r in frame.itertuples(index=False)]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as im:
            img = im.convert("RGB")
        return self.transform(img), label

parts = {s: clean[clean["split"] == s].reset_index(drop=True)
         for s in ("train", "val", "test")}
NUM_WORKERS = 0 if os.name == "nt" else 2      # Windows: notebook classes can't spawn
PIN = DEVICE.type == "cuda"
GEN = torch.Generator(); GEN.manual_seed(SEED)

def make_loader(frame, tf, shuffle):
    return DataLoader(ManifestDataset(frame, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=PIN, generator=GEN)

train_loader = make_loader(parts["train"], train_tf, True)
train_eval_loader = make_loader(parts["train"], eval_tf, False)
val_loader = make_loader(parts["val"], eval_tf, False)
test_loader = make_loader(parts["test"], eval_tf, False)

cw = json.loads((OUT_DIR / "class_weights.json").read_text())
CLASS_WEIGHTS = (torch.tensor([cw[c] for c in CLASS_NAMES], dtype=torch.float32)
                 .to(DEVICE) if USE_CLASS_WEIGHTS else None)

print(f"manifest {MANIFEST}")
print(f"device {DEVICE.type} | train {len(parts['train'])} val {len(parts['val'])} "
      f"test {len(parts['test'])} | {N_CLASSES} classes | workers {NUM_WORKERS}")

## 2. The model

In [ ]:
# ---- model: LiteDiseaseNetV4 ----------------------------------------------- #
class ChannelAttention(nn.Module):
    """CBAM channel attention: shared MLP over avg- and max-pooled descriptors."""
    def __init__(self, ch, reduction=16):
        super().__init__()
        hidden = max(ch // reduction, 8)
        self.mlp = nn.Sequential(nn.Linear(ch, hidden), nn.ReLU(inplace=True),
                                 nn.Linear(hidden, ch))
    def forward(self, x):
        b, c, _, _ = x.shape
        avg = self.mlp(x.mean(dim=(2, 3)))
        mx = self.mlp(x.amax(dim=(2, 3)))
        return x * torch.sigmoid(avg + mx).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    """CBAM spatial attention: 7x7 conv over channel-avg and channel-max maps."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
    def forward(self, x):
        m = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        return x * torch.sigmoid(self.conv(m))

class CBAM(nn.Module):
    def __init__(self, ch, reduction=16):
        super().__init__()
        self.channel = ChannelAttention(ch, reduction)
        self.spatial = SpatialAttention()
    def forward(self, x):
        return self.spatial(self.channel(x))

class ResidualCBAMBlock(nn.Module):
    """Bottleneck refine block with CBAM, residual add. Replaces V3's ResidualSEBlock."""
    def __init__(self, ch, bottleneck=192):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(ch, bottleneck, 1, bias=False), nn.BatchNorm2d(bottleneck),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck, bottleneck, 3, padding=1, bias=False),
            nn.BatchNorm2d(bottleneck), nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck, ch, 1, bias=False), nn.BatchNorm2d(ch),
        )
        self.attn = CBAM(ch)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.act(x + self.attn(self.body(x)))

class LiteDiseaseNetV4(nn.Module):
    TEX_CH, SEM_CH, HIDDEN = 96, 1280, 768

    def __init__(self, n_classes, pretrained=True):
        super().__init__()
        weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
        layers = mobilenet_v2(weights=weights).features
        self.stem = layers[:7]                 # 32ch @ 28x28 - texture branch input
        self.stage2 = layers[7:]               # -> 1280ch @ 7x7 - semantic branch

        # texture branch: was SE + pool on raw features; now a small conv bottleneck
        self.texture = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1, bias=False), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, self.TEX_CH, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(self.TEX_CH), nn.ReLU(inplace=True),
            CBAM(self.TEX_CH),
        )
        # semantic refine: 2 blocks (V3 had 1)
        self.refine = nn.Sequential(ResidualCBAMBlock(self.SEM_CH),
                                    ResidualCBAMBlock(self.SEM_CH))
        self.pool = nn.AdaptiveAvgPool2d(1)
        # learned fusion gate: per-image weights for the two branches
        self.gate = nn.Linear(self.SEM_CH + self.TEX_CH, 2)
        self.classifier = nn.Sequential(
            nn.Linear(self.SEM_CH + self.TEX_CH, self.HIDDEN),
            nn.BatchNorm1d(self.HIDDEN), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(self.HIDDEN, n_classes),
        )

    def forward(self, x):
        low = self.stem(x)
        tex = self.pool(self.texture(low)).flatten(1)          # B x 96
        sem = self.pool(self.refine(self.stage2(low))).flatten(1)   # B x 1280
        both = torch.cat([sem, tex], dim=1)                    # B x 1376
        g = torch.softmax(self.gate(both), dim=1)              # B x 2
        fused = torch.cat([sem * (2 * g[:, :1]), tex * (2 * g[:, 1:])], dim=1)
        return self.classifier(fused)

    def backbone_parameters(self):
        return [p for m in (self.stem, self.stage2) for p in m.parameters()]

    def head_parameters(self):
        return [p for m in (self.texture, self.refine, self.gate, self.classifier)
                for p in m.parameters()]

MODEL = LiteDiseaseNetV4(N_CLASSES).to(DEVICE)
n_total = sum(p.numel() for p in MODEL.parameters())
n_backbone = sum(p.numel() for p in MODEL.backbone_parameters())
print(f"LiteDiseaseNetV4: {n_total:,} params "
      f"(backbone {n_backbone:,}, new modules {n_total - n_backbone:,})")
with torch.no_grad():
    out = MODEL(torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
assert out.shape == (2, N_CLASSES), out.shape
print("forward pass OK", tuple(out.shape))

## 3. Train

`run_epoch` does both directions: pass an optimizer and it trains, leave it out and it
evaluates under `no_grad`. Two parameter groups carry the two learning rates. Per-epoch
numbers stream to `litedisease_v4_log.csv` after every epoch, so an interrupted run
still leaves the curve behind, and the best checkpoint by val macro-F1 goes to
`litedisease_v4_best.pth`.

In [ ]:
# ---- train ------------------------------------------------------------------ #
CRITERION = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=LABEL_SMOOTHING)
OPTIMIZER = torch.optim.AdamW(
    [{"params": MODEL.backbone_parameters(), "lr": LR_BACKBONE},
     {"params": MODEL.head_parameters(), "lr": LR_HEAD}],
    weight_decay=WEIGHT_DECAY)
SCHEDULER = torch.optim.lr_scheduler.ReduceLROnPlateau(
    OPTIMIZER, mode="max", factor=0.5, patience=3)
SCALER = torch.amp.GradScaler(enabled=AMP_ON)

LOG_PATH = OUT_DIR / "litedisease_v4_log.csv"
BEST_PATH = OUT_DIR / "litedisease_v4_best.pth"

def run_epoch(model, loader, optimizer=None, desc=""):
    training = optimizer is not None
    model.train(training)
    tot_loss = n_seen = 0
    preds, targets = [], []
    for x, y in tqdm(loader, desc=desc, leave=False, unit="b"):
        x, y = x.to(DEVICE, non_blocking=PIN), y.to(DEVICE, non_blocking=PIN)
        with torch.set_grad_enabled(training):
            with torch.autocast(DEVICE.type, enabled=AMP_ON):
                out = model(x)
                loss = CRITERION(out, y)
        if training:
            optimizer.zero_grad(set_to_none=True)
            SCALER.scale(loss).backward()
            SCALER.step(optimizer)
            SCALER.update()
        tot_loss += loss.item() * len(y)
        n_seen += len(y)
        preds.append(out.argmax(1).cpu())
        targets.append(y.cpu())
    preds, targets = torch.cat(preds).numpy(), torch.cat(targets).numpy()
    p, r, f1, _ = precision_recall_fscore_support(
        targets, preds, average="macro", zero_division=0)
    return {"loss": tot_loss / n_seen, "accuracy": accuracy_score(targets, preds),
            "macro_precision": p, "macro_recall": r, "macro_f1": f1,
            "n": n_seen, "preds": preds, "targets": targets}

best = {"epoch": 0, "val_macro_f1": -1.0}
rows = []
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = run_epoch(MODEL, train_loader, OPTIMIZER, desc=f"train {epoch}/{EPOCHS}")
    va = run_epoch(MODEL, val_loader, desc=f"val {epoch}/{EPOCHS}")
    SCHEDULER.step(va["macro_f1"])
    rows.append({"epoch": epoch,
                 "lr_backbone": OPTIMIZER.param_groups[0]["lr"],
                 "lr_head": OPTIMIZER.param_groups[1]["lr"],
                 "seconds": round(time.time() - t0, 1),
                 "train_loss": tr["loss"], "train_acc": tr["accuracy"],
                 "train_macro_f1": tr["macro_f1"],
                 "val_loss": va["loss"], "val_acc": va["accuracy"],
                 "val_macro_f1": va["macro_f1"]})
    pd.DataFrame(rows).to_csv(LOG_PATH, index=False)
    star = ""
    if va["macro_f1"] > best["val_macro_f1"]:
        best = {"epoch": epoch, "val_macro_f1": va["macro_f1"]}
        torch.save(MODEL.state_dict(), BEST_PATH)
        star = "  <- best"
    print(f"epoch {epoch:>2}/{EPOCHS}  train loss {tr['loss']:.4f} acc {tr['accuracy']:.4f} "
          f"| val loss {va['loss']:.4f} acc {va['accuracy']:.4f} "
          f"macro-F1 {va['macro_f1']:.4f}{star}")
print(f"\nbest: epoch {best['epoch']} val macro-F1 {best['val_macro_f1']:.4f}")

## 4. Evaluate the best checkpoint, save results and figures

In [ ]:
# ---- evaluate the best checkpoint ------------------------------------------- #
MODEL.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE, weights_only=True))
print(f"loaded {BEST_PATH} (epoch {best['epoch']})\n")

store = {}
for split, ld in (("train", train_eval_loader), ("val", val_loader), ("test", test_loader)):
    r = store[split] = run_epoch(MODEL, ld, desc=f"eval {split}")
    print(f"{split:<6} n={r['n']:<6} loss {r['loss']:.4f}  acc {r['accuracy']:.4f}  "
          f"macro-P {r['macro_precision']:.4f}  macro-R {r['macro_recall']:.4f}  "
          f"macro-F1 {r['macro_f1']:.4f}")

print("\n" + classification_report(store["test"]["targets"], store["test"]["preds"],
                                    target_names=CLASS_NAMES, digits=4, zero_division=0))

RESULTS = {
    "model": "litedisease_v4", "arch": "LiteDiseaseNetV4 (mobilenet_v2 dual-branch)",
    "params_total": int(n_total), "seed": SEED, "epochs": EPOCHS,
    "lr_backbone": LR_BACKBONE, "lr_head": LR_HEAD,
    "label_smoothing": LABEL_SMOOTHING, "batch_size": BATCH_SIZE,
    "img_size": IMG_SIZE, "class_weighted": CLASS_WEIGHTS is not None,
    "amp": bool(AMP_ON), "best_epoch": int(best["epoch"]),
    "best_val_macro_f1": round(float(best["val_macro_f1"]), 6),
    "n_classes": N_CLASSES, "class_names": CLASS_NAMES,
    "data_source": "clean (repaired in Part A of crop_disease_full_pipeline.ipynb)",
    "splits": {s: {k: (round(float(v), 6) if isinstance(v, float) else int(v))
                   for k, v in r.items() if k in
                   ("n", "loss", "accuracy", "macro_precision", "macro_recall", "macro_f1")}
               for s, r in store.items()},
}
(OUT_DIR / "litedisease_v4_results.json").write_text(json.dumps(RESULTS, indent=2))

torch.save({"state_dict": MODEL.state_dict(), "class_names": CLASS_NAMES,
            "img_size": IMG_SIZE, "arch": "LiteDiseaseNetV4",
            "model_key": "litedisease_v4",
            "normalize_mean": MEAN, "normalize_std": STD,
            "test_metrics": RESULTS["splits"]["test"]},
           OUT_DIR / "crop_disease_litedisease_v4.pth")
print("saved litedisease_v4_results.json and crop_disease_litedisease_v4.pth")

log = pd.read_csv(LOG_PATH)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(log["epoch"], log["train_macro_f1"], label="train")
axes[1].plot(log["epoch"], log["val_macro_f1"], label="val")
axes[1].axvline(best["epoch"], ls=":", c="k", label=f"best (ep {best['epoch']})")
axes[1].set_title("macro-F1"); axes[1].set_xlabel("epoch"); axes[1].legend()
fig.suptitle("LiteDiseaseNetV4"); fig.tight_layout()
fig.savefig(OUT_DIR / "litedisease_v4_curves.png", dpi=120)
plt.show()

cm = confusion_matrix(store["test"]["targets"], store["test"]["preds"])
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
short = [c.replace("___", " ").replace("__", " ").replace("_", " ")[:24]
         for c in CLASS_NAMES]
ax.set_xticklabels(short, rotation=90, fontsize=7)
ax.set_yticklabels(short, fontsize=7)
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title("LiteDiseaseNetV4 - test confusion matrix")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
fig.tight_layout()
fig.savefig(OUT_DIR / "litedisease_v4_confusion_test.png", dpi=120)
plt.show()

## 5. Test-time augmentation

Averaging the prediction with its horizontal-flip counterpart costs 2x inference and no
retraining. Leaves have no canonical orientation, so the mirror is a genuinely free
second opinion. The `test_tta` block is written back into the results file beside the
plain numbers - report whichever you use, but say which.

In [ ]:
# ---- test-time augmentation: average plain + horizontal-flip predictions ---- #
# Free accuracy: no retraining, 2x inference cost. Uses the best checkpoint
# already loaded in the evaluation cell above.
MODEL.eval()
plain_probs, tta_probs, targets_l = [], [], []
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="TTA test", leave=False, unit="b"):
        x = x.to(DEVICE)
        p1 = MODEL(x).softmax(1)
        p2 = MODEL(torch.flip(x, dims=[3])).softmax(1)
        plain_probs.append(p1.cpu()); tta_probs.append((p1 + p2).cpu())
        targets_l.append(y)

targets_np = torch.cat(targets_l).numpy()
plain_pred = torch.cat(plain_probs).argmax(1).numpy()
tta_pred = torch.cat(tta_probs).argmax(1).numpy()

for name, pred in (("plain", plain_pred), ("TTA", tta_pred)):
    acc = accuracy_score(targets_np, pred)
    p, r, f1, _ = precision_recall_fscore_support(targets_np, pred,
                                                  average="macro", zero_division=0)
    print(f"{name:6} acc {acc*100:.2f}%  macro-P {p*100:.2f}%  "
          f"macro-R {r*100:.2f}%  macro-F1 {f1*100:.2f}%")
fixed = int(((plain_pred != targets_np) & (tta_pred == targets_np)).sum())
broken = int(((plain_pred == targets_np) & (tta_pred != targets_np)).sum())
print(f"TTA fixed {fixed} predictions, broke {broken}")

# record next to the plain numbers so the results file carries both
p, r, f1, _ = precision_recall_fscore_support(targets_np, tta_pred,
                                              average="macro", zero_division=0)
RESULTS["splits"]["test_tta"] = {
    "n": int(len(targets_np)),
    "accuracy": round(float(accuracy_score(targets_np, tta_pred)), 6),
    "macro_precision": round(float(p), 6), "macro_recall": round(float(r), 6),
    "macro_f1": round(float(f1), 6),
}
(OUT_DIR / "litedisease_v4_results.json").write_text(json.dumps(RESULTS, indent=2))
print("test_tta block written to litedisease_v4_results.json")

## 6. Compare against every other run in the folder

In [ ]:
# ---- every results json in OUT_DIR, ranked by test macro-F1 ---------------- #
found = []
for p in sorted(OUT_DIR.glob("*results*.json")):
    try:
        r = json.loads(Path(p).read_text())
    except Exception as e:
        print(f"skipping unreadable {p.name}: {e}")
        continue
    t = r["splits"]["test"]
    row = {"model": r.get("model", p.stem), "params": r.get("params_total", 0),
           "test_acc": round(t["accuracy"], 4), "test_macro_f1": round(t["macro_f1"], 4),
           "best_epoch": r.get("best_epoch"), "n_classes": r.get("n_classes")}
    if "test_tta" in r["splits"]:
        row["test_macro_f1_tta"] = round(r["splits"]["test_tta"]["macro_f1"], 4)
    found.append(row)

if not found:
    print(f"no results json in {OUT_DIR}")
else:
    table = (pd.DataFrame(found).sort_values("test_macro_f1", ascending=False)
             .reset_index(drop=True))
    table["params"] = table["params"].map(lambda n: f"{n:,}")
    print(table.to_string(index=False))
    table.to_csv(OUT_DIR / "model_comparison.csv", index=False)
    print(f"\nsaved {OUT_DIR / 'model_comparison.csv'}")
    print("\nnote: recipes differ between notebooks (epochs, LR schedule, label")
    print("smoothing), so this ranks the builds as delivered - it is not a controlled")
    print("architecture-only comparison.")

## 7. Try it on your own photo

Copy photos into `my_photos/` next to this notebook and run the cell. Predictions use
flip-TTA, the strongest configuration here.

**Confidence gate.** The model has no "none of these" option - softmax always sums to 1,
so it must name a class even for a crop it never saw. Anything below the gate is
reported as UNCERTAIN rather than asserted as a diagnosis. The threshold is calibrated
from this model's own validation predictions: the 5th percentile of confidence on the
ones it got right.

Label smoothing is what makes this work. Measured on this project, a cross-entropy
backbone rated out-of-scope photos *above* the 5th percentile of its genuine
predictions - no threshold could separate them. This model rates the same photos far
lower, so the gate has something to cut on.

In [ ]:
CONFIDENCE_THRESHOLD = None    # None -> 5th percentile of correct val predictions

# ---- calibrate on the validation split ------------------------------------- #
MODEL.eval()
_vp, _vt = [], []
with torch.no_grad():
    for _x, _y in tqdm(val_loader, desc="calibrating", leave=False, unit="b"):
        _x = _x.to(DEVICE)
        _p = (MODEL(_x).softmax(1) + MODEL(torch.flip(_x, dims=[3])).softmax(1)) / 2
        _vp.append(_p.cpu())
        _vt.append(_y)
_vp, _vt = torch.cat(_vp).numpy(), torch.cat(_vt).numpy()
_ok = _vp.argmax(1) == _vt
if CONFIDENCE_THRESHOLD is None:
    CONFIDENCE_THRESHOLD = float(np.percentile(_vp.max(1)[_ok], 5))
print(f"gate = {CONFIDENCE_THRESHOLD:.3f}  (keeps "
      f"{(_vp.max(1) >= CONFIDENCE_THRESHOLD).mean():.1%} of validation images)")

# ---- classify everything in my_photos/ ------------------------------------- #
PHOTO_DIR = Path("my_photos")
PHOTO_DIR.mkdir(exist_ok=True)
_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
_photos = sorted(p for p in PHOTO_DIR.iterdir()
                 if p.suffix.lower() in _exts and not p.name.startswith("prediction"))
if not _photos:
    print(f"\nNo photos found. Copy leaf photos into {PHOTO_DIR.resolve()} and rerun.")

_short = [c.replace("___", " ").replace("__", " ").replace("_", " ")[:30]
          for c in CLASS_NAMES]
for _f in _photos:
    try:
        _img = Image.open(_f).convert("RGB")
    except Exception as e:
        print(f"{_f.name}: not an image ({e})")
        continue
    _x = eval_tf(_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _prob = (MODEL(_x).softmax(1) + MODEL(torch.flip(_x, dims=[3])).softmax(1))[0]
        _prob = (_prob / _prob.sum()).cpu().numpy()
    _top = np.argsort(_prob)[::-1][:5]

    _fig, _ax = plt.subplots(1, 2, figsize=(12, 4.6),
                             gridspec_kw={"width_ratios": [1, 1.5]})
    _ax[0].imshow(_img)
    _ax[0].axis("off")
    _ax[0].set_title(_f.name[:40])
    _ax[1].barh([_short[i] for i in _top][::-1], _prob[_top][::-1],
                color=["#c7d9f1"] * 4 + ["#1f6fb4"])
    _ax[1].set_xlim(0, 1)
    _ax[1].set_xlabel("probability")
    _ax[1].set_title(f"LiteDiseaseNetV4+TTA: {CLASS_NAMES[_top[0]]} ({_prob[_top[0]]:.1%})")
    for _i, _v in enumerate(_prob[_top][::-1]):
        _ax[1].text(_v + 0.01, _i, f"{_v:.1%}", va="center", fontsize=9)
    _fig.tight_layout()
    plt.show()

    if _prob[_top[0]] >= CONFIDENCE_THRESHOLD:
        print(f"{_f.name} -> {CLASS_NAMES[_top[0]]}  {_prob[_top[0]]:.1%}")
    else:
        print(f"{_f.name} -> UNCERTAIN (best guess {CLASS_NAMES[_top[0]]} at "
              f"{_prob[_top[0]]:.1%}, below the {CONFIDENCE_THRESHOLD:.1%} gate)")
        print("  likely a crop or disease outside the trained classes, or a field "
              "photo unlike the studio training images")